# Emergency Lane Occupancy Detection System

This notebook implements an **AI-Based Computer Vision System** to detect and track unauthorized vehicles occupying emergency lanes. 

### Core Features:
1. **Object Detection & Tracking**: Employs YOLO (v8/v11) with ByteTrack/BoT-SORT to detect and track vehicles.
2. **Dynamic Region of Interest (ROI)**: Supports multi-point polygon coordinates representing the emergency lane.
3. **Temporal Verification**: Filters out transient crossings by requiring vehicles to stay inside the lane for a minimum number of frames before generating an alert.
4. **Alerts & Evidence Snapshots**: Saves cropped and full snapshots of violating vehicles in the `evidence/` directory.
5. **CSV Event Log**: Logs entry/exit timestamps, durations, and details of all violations to a CSV file.

In [ ]:
# Install dependencies
# %pip install ultralytics opencv-python shapely pandas numpy matplotlib tqdm ipywidgets

## 1. Configuration & Parameters

Configure the inputs, outputs, thresholds, and emergency lane coordinates below.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from shapely.geometry import Polygon, Point
from shapely.ops import unary_union
from ultralytics import YOLO
from datetime import datetime
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# --- CONFIGURATION SETTINGS ---

# Paths
VIDEO_PATH = "traffic_video.mp4"         # Replace with your test video path
OUTPUT_VIDEO_PATH = "output_annotated.mp4" # Path for output video
EVIDENCE_DIR = "evidence"                 # Folder to save violation snapshots
REPORT_PATH = "occupancy_report.csv"      # Path to save CSV logs
FRAME_LOG_PATH = "frame_occupancy_log.csv" # Path to save frame occupancy logs

# Detection and Tracking Settings
MODEL_NAME = "yolov8s.pt"                 # YOLO model size (n, s, m, l, x)
CONFIDENCE_THRESHOLD = 0.40              # Minimum detection confidence
CLASS_IDS_TO_TRACK = [2, 3, 5, 7]        # COCO IDs: 2=car, 3=motorcycle, 5=bus, 7=truck
TRACKER_TYPE = "bytetrack.yaml"          # ByteTrack (default) or 'botsort.yaml'

# Emergency Lane Polygon Coordinates (ROI)
# List of (x, y) coordinates representing the polygon of the emergency lane.
ROI_POLYGON_COORDS = [
    (100, 720),   # Bottom-left
    (400, 300),   # Top-left
    (600, 300),   # Top-right
    (300, 720)    # Bottom-right
]

# Validation & Trigger Settings
MIN_VIOLATION_FRAMES = 15                 # Consecutive frames inside ROI to trigger a confirmed violation
FRAME_SAMPLING_RATE = 1                  # Process every Nth frame

# Occupancy Thresholds & Smoothing
LOW_OCCUPANCY_THRESHOLD = 15.0           # Occupancy % below this is considered Low/Clear traffic
MEDIUM_OCCUPANCY_THRESHOLD = 45.0        # Occupancy % below this is Medium traffic, above is High traffic
SMOOTHING_WINDOW_SIZE = 15               # Number of frames to smooth occupancy values over

# Create folders if they do not exist
os.makedirs(EVIDENCE_DIR, exist_ok=True)

print("Configuration loaded. Evidence snapshots directory verified/created.")

## 2. ROI Selection Helper Tools

To run this pipeline, you need the polygon coordinates of the emergency lane in your video. Choose one of the two tools below:

### Tool A: Interactive OpenCV ROI Selector (Requires GUI/Local Machine)
Run the cell below to load the first frame of your video, click the vertices of the emergency lane in order (clockwise or counter-clockwise), and press **Enter** when done. Press **'r'** to reset the points.

In [ ]:
def select_roi(video_path):
    if not os.path.exists(video_path):
        print(f"Error: Video file '{video_path}' not found. Please place your video and adjust VIDEO_PATH.")
        return None
        
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    
    if not ret:
        print("Error: Could not read frame from video.")
        return None
        
    pts = []
    window_name = "Select Emergency Lane ROI (Press Enter when done, 'r' to reset)"
    
    def click_event(event, x, y, flags, params):
        if event == cv2.EVENT_LBUTTONDOWN:
            pts.append((x, y))
            cv2.circle(img_copy, (x, y), 5, (0, 0, 255), -1)
            if len(pts) > 1:
                cv2.line(img_copy, pts[-2], pts[-1], (0, 255, 0), 2)
            cv2.imshow(window_name, img_copy)
            
    img_copy = frame.copy()
    cv2.imshow(window_name, img_copy)
    cv2.setMouseCallback(window_name, click_event)
    
    while True:
        key = cv2.waitKey(1) & 0xFF
        if key == 13: # Enter key
            if len(pts) > 2:
                cv2.line(img_copy, pts[-1], pts[0], (0, 255, 0), 2)
                cv2.imshow(window_name, img_copy)
                cv2.waitKey(1000)
            break
        elif key == ord('r'): # Reset
            pts = []
            img_copy = frame.copy()
            cv2.imshow(window_name, img_copy)
            
    cv2.destroyAllWindows()
    print("Selected ROI Coordinates:")
    print(pts)
    return pts

# To run the selector, uncomment the following line and execute:
# selected_coords = select_roi(VIDEO_PATH)

### Tool B: Static Matplotlib ROI Coordinator (Works on JupyterHub / Headless Servers)
If you are running the notebook on a server without graphical windows, run the cell below. It will plot the first frame of the video with grid lines. You can read the $(x, y)$ coordinates from the axes grid to define your `ROI_POLYGON_COORDS`.

In [ ]:
def plot_static_frame(video_path):
    if not os.path.exists(video_path):
        print(f"Error: Video file '{video_path}' not found.")
        return
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        print("Error: Could not read frame from video.")
        return
    
    # Convert BGR to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(12, 7))
    plt.imshow(frame_rgb)
    plt.title("Static Frame - Use grid to estimate ROI polygon coordinates")
    plt.grid(True)
    plt.xlabel("X coordinate")
    plt.ylabel("Y coordinate")
    plt.show()

# To run the static frame viewer, uncomment the line below:
# plot_static_frame(VIDEO_PATH)

## 3. Core Logic: Emergency Lane Detector

This class handles object detection, object tracking across frames, ROI polygon containment check (using the bottom-center of the vehicle's bounding box), visual overlay rendering, evidence snapshot saving, and event logging.

In [ ]:
class EmergencyLaneDetector:
    def __init__(self, model_name, roi_coords, min_violation_frames=15, conf=0.40, classes=[2,3,5,7],
                 low_threshold=15.0, medium_threshold=45.0, smoothing_window=15):
        """
        Initialize the detector with YOLO model and ROI polygon.
        """
        self.model = YOLO(model_name)
        self.roi_coords = roi_coords
        self.min_violation_frames = min_violation_frames
        self.conf = conf
        self.classes = classes
        
        # Thresholds & Smoothing
        self.low_threshold = low_threshold
        self.medium_threshold = medium_threshold
        self.smoothing_window = smoothing_window
        self.occupancy_history = []
        
        # Define shapely polygon for occupancy check
        self.roi_polygon = Polygon(roi_coords)
        self.roi_area = self.roi_polygon.area
        
        # State tracking: track_id -> dict of status
        self.track_states = {}
        
        # List of final violation records for CSV export
        self.violation_records = []
        
        # List of frame occupancy records for CSV export
        self.frame_records = []
        
    def is_inside_roi(self, bbox):
        """
        Checks if the bottom-center point of the bounding box is inside the ROI polygon.
        bbox format: [x1, y1, x2, y2]
        """
        x1, y1, x2, y2 = bbox
        bottom_center = ((x1 + x2) / 2.0, y2)
        point = Point(bottom_center)
        return self.roi_polygon.contains(point)
        
    def process_frame(self, frame, frame_num, fps):
        """
        Processes a single video frame. Detects/tracks objects, checks for violations,
        annotates the frame, and manages the violation states.
        """
        annotated_frame = frame.copy()
        
        # Run YOLO tracking on the frame with persistence enabled
        results = self.model.track(
            source=frame, 
            persist=True, 
            conf=self.conf, 
            classes=self.classes, 
            tracker=TRACKER_TYPE,
            verbose=False
        )
        
        # Track whether the lane is currently occupied by any violating vehicle
        lane_occupied = False
        active_track_ids_in_frame = set()
        
        # List to hold intersection polygons for area occupancy calculation
        intersection_polys = []
        vehicles_in_roi_details = []
        
        # Parse tracking results
        if results and results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            track_ids = results[0].boxes.id.cpu().numpy().astype(int)
            class_indices = results[0].boxes.cls.cpu().numpy().astype(int)
            confidences = results[0].boxes.conf.cpu().numpy()
            
            names = self.model.names
            
            for bbox, track_id, cls_idx, confidence in zip(boxes, track_ids, class_indices, confidences):
                active_track_ids_in_frame.add(track_id)
                class_name = names[cls_idx]
                
                # Check for physical overlap with ROI (for occupancy percentage)
                x1, y1, x2, y2 = bbox
                vehicle_poly = Polygon([(x1, y1), (x2, y1), (x2, y2), (x1, y2)])
                intersection = self.roi_polygon.intersection(vehicle_poly)
                
                # We require at least 5% of the vehicle to overlap the lane, or a significant area,
                # to filter out boundary-touching vehicles from adjacent lanes
                if not intersection.is_empty and intersection.area > 0.05 * vehicle_poly.area:
                    intersection_polys.append(intersection)
                    vehicles_in_roi_details.append((track_id, class_name))
                
                # Check if bottom-center of vehicle is inside the emergency lane ROI (for violations)
                inside = self.is_inside_roi(bbox)
                
                if inside:
                    # Update tracking states
                    if track_id not in self.track_states:
                        self.track_states[track_id] = {
                            'consecutive_frames_in_roi': 1,
                            'violation_started': False,
                            'entry_time': None,
                            'frame_entered': frame_num,
                            'last_seen_frame': frame_num,
                            'class_name': class_name,
                            'snapshot_taken': False
                        }
                    else:
                        self.track_states[track_id]['consecutive_frames_in_roi'] += 1
                        self.track_states[track_id]['last_seen_frame'] = frame_num
                    
                    consecutive_frames = self.track_states[track_id]['consecutive_frames_in_roi']
                    
                    # If vehicle is in the lane for more than the threshold, confirm violation
                    if consecutive_frames >= self.min_violation_frames:
                        lane_occupied = True
                        
                        if not self.track_states[track_id]['violation_started']:
                            self.track_states[track_id]['violation_started'] = True
                            timestamp_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
                            self.track_states[track_id]['entry_time'] = timestamp_str
                            print(f"[ALERT] Frame {frame_num}: Vehicle (ID: {track_id}, Type: {class_name}) has occupied the emergency lane!")
                            
                        # Save snapshot if not already done
                        if not self.track_states[track_id]['snapshot_taken']:
                            self.save_evidence_snapshot(frame, bbox, track_id, class_name, frame_num)
                            self.track_states[track_id]['snapshot_taken'] = True
                            
                    # Draw bounding box for violating or potential violating vehicle
                    color = (0, 0, 255) if self.track_states[track_id]['violation_started'] else (0, 165, 255) # Red for confirmed, Orange for checking
                    x1_i, y1_i, x2_i, y2_i = bbox.astype(int)
                    cv2.rectangle(annotated_frame, (x1_i, y1_i), (x2_i, y2_i), color, 2)
                    
                    status_text = "VIOLATION" if self.track_states[track_id]['violation_started'] else "Checking..."
                    cv2.putText(
                        annotated_frame, 
                        f"ID: {track_id} {class_name} ({status_text})", 
                        (x1_i, max(y1_i - 10, 20)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 
                        0.5, 
                        color, 
                        2,
                        cv2.LINE_AA
                    )
                else:
                    # Vehicle is outside ROI
                    # If it was in the ROI previously, check if it was violating and now exited
                    if track_id in self.track_states:
                        if self.track_states[track_id]['violation_started']:
                            self.record_violation_event(track_id, frame_num, fps)
                        del self.track_states[track_id]
                        
                    # Draw standard YOLO box for normal traffic outside emergency lane
                    x1_i, y1_i, x2_i, y2_i = bbox.astype(int)
                    cv2.rectangle(annotated_frame, (x1_i, y1_i), (x2_i, y2_i), (255, 255, 0), 1)
                    cv2.putText(
                        annotated_frame, 
                        f"ID: {track_id} {class_name}", 
                        (x1_i, max(y1_i - 10, 20)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 
                        0.5, 
                        (255, 255, 0), 
                        1,
                        cv2.LINE_AA
                    )
                    
        # Check for vehicles that left the frame entirely
        tracks_to_remove = []
        for track_id, state in self.track_states.items():
            if track_id not in active_track_ids_in_frame:
                if state['violation_started']:
                    self.record_violation_event(track_id, frame_num, fps)
                tracks_to_remove.append(track_id)
                
        for track_id in tracks_to_remove:
            if track_id in self.track_states:
                del self.track_states[track_id]
                
        # --- OCCUPANCY & SMOOTHING MATH ---
        # Compute raw occupancy based on the union of all intersections
        if intersection_polys:
            union_poly = unary_union(intersection_polys)
            occupied_area = union_poly.area
        else:
            occupied_area = 0.0
            
        raw_occupancy = (occupied_area / self.roi_area) * 100.0
        
        # Smooth occupancy values using moving average
        self.occupancy_history.append(raw_occupancy)
        if len(self.occupancy_history) > self.smoothing_window:
            self.occupancy_history.pop(0)
        smoothed_occupancy = sum(self.occupancy_history) / len(self.occupancy_history)
        
        # Classify traffic density level and set color
        if smoothed_occupancy < self.low_threshold:
            traffic_level = "Low"
            poly_color = (0, 255, 0)      # Green
        elif smoothed_occupancy < self.medium_threshold:
            traffic_level = "Medium"
            poly_color = (0, 165, 255)    # Orange
        else:
            traffic_level = "High"
            poly_color = (0, 0, 255)      # Red
            
        # Log frame occupancy details
        timestamp_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
        self.frame_records.append({
            'Frame': frame_num,
            'Timestamp': timestamp_str,
            'Raw Occupancy (%)': round(raw_occupancy, 2),
            'Smoothed Occupancy (%)': round(smoothed_occupancy, 2),
            'Traffic Level': traffic_level,
            'Vehicles in ROI': len(vehicles_in_roi_details)
        })
        
        # --- VISUAL RENDERING ---
        # Draw ROI Polygon overlay (translucent with color corresponding to traffic level)
        overlay = annotated_frame.copy()
        cv2.fillPoly(overlay, [np.array(self.roi_coords, dtype=np.int32)], poly_color)
        cv2.addWeighted(overlay, 0.25, annotated_frame, 0.75, 0, annotated_frame)
        cv2.polylines(annotated_frame, [np.array(self.roi_coords, dtype=np.int32)], True, poly_color, 2)
        
        # Draw Premium HUD
        hud_x1, hud_y1 = 20, 20
        hud_x2, hud_y2 = 480, 190
        hud_overlay = annotated_frame.copy()
        cv2.rectangle(hud_overlay, (hud_x1, hud_y1), (hud_x2, hud_y2), (0, 0, 0), -1)
        cv2.addWeighted(hud_overlay, 0.65, annotated_frame, 0.35, 0, annotated_frame)
        cv2.rectangle(annotated_frame, (hud_x1, hud_y1), (hud_x2, hud_y2), (180, 180, 180), 1)
        
        # Title
        cv2.putText(annotated_frame, "EMERGENCY LANE MONITOR", (hud_x1 + 15, hud_y1 + 25), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        
        # Occupancy text & progress bar
        cv2.putText(annotated_frame, f"Occupancy: {smoothed_occupancy:.1f}%", (hud_x1 + 15, hud_y1 + 58), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        
        # Draw progress bar background
        bar_x1, bar_y1 = hud_x1 + 150, hud_y1 + 48
        bar_w, bar_h = 280, 12
        cv2.rectangle(annotated_frame, (bar_x1, bar_y1), (bar_x1 + bar_w, bar_y1 + bar_h), (50, 50, 50), -1)
        # Draw progress bar fill
        fill_w = int(bar_w * (smoothed_occupancy / 100.0))
        fill_w = min(bar_w, max(0, fill_w))
        cv2.rectangle(annotated_frame, (bar_x1, bar_y1), (bar_x1 + fill_w, bar_y1 + bar_h), poly_color, -1)
        
        # Traffic Level text
        cv2.putText(annotated_frame, "Traffic Density:", (hud_x1 + 15, hud_y1 + 90), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        
        # Draw level badge
        badge_text = traffic_level.upper()
        (tw, th), _ = cv2.getTextSize(badge_text, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        badge_x1, badge_y1 = hud_x1 + 150, hud_y1 + 77
        badge_x2, badge_y2 = badge_x1 + tw + 20, badge_y1 + th + 8
        cv2.rectangle(annotated_frame, (badge_x1, badge_y1), (badge_x2, badge_y2), poly_color, -1)
        # Medium level (yellowish orange) has black text for better contrast, others have white text
        badge_txt_color = (0, 0, 0) if traffic_level == "Medium" else (255, 255, 255)
        cv2.putText(annotated_frame, badge_text, (badge_x1 + 10, badge_y1 + th + 4), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, badge_txt_color, 1, cv2.LINE_AA)
        
        # Vehicles inside ROI
        vehicles_text = f"Vehicles in ROI: {len(vehicles_in_roi_details)}"
        if vehicles_in_roi_details:
            details_str = ", ".join([f"ID {vid}({vcls})" for vid, vcls in vehicles_in_roi_details[:2]])
            if len(vehicles_in_roi_details) > 2:
                details_str += "..."
            vehicles_text += f" ({details_str})"
            
        cv2.putText(annotated_frame, vehicles_text, (hud_x1 + 15, hud_y1 + 122), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        
        # Violation status
        violation_status = "ACTIVE VIOLATION" if lane_occupied else "NO VIOLATION"
        v_color = (0, 0, 255) if lane_occupied else (0, 255, 0)
        cv2.putText(annotated_frame, f"Status: {violation_status}", (hud_x1 + 15, hud_y1 + 154), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, v_color, 2, cv2.LINE_AA)
        
        return annotated_frame

    def save_evidence_snapshot(self, frame, bbox, track_id, class_name, frame_num):
        """
        Saves a snapshot of the frame and a cropped image of the violating vehicle.
        """
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        x1, y1, x2, y2 = bbox.astype(int)
        h, w = frame.shape[:2]
        
        # Ensure coordinates are within frame boundaries
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        
        # Save crop of the vehicle
        crop = frame[y1:y2, x1:x2]
        if crop.size > 0:
            crop_filename = f"crop_vehicle_{track_id}_{timestamp}.jpg"
            crop_path = os.path.join(EVIDENCE_DIR, crop_filename)
            cv2.imwrite(crop_path, crop)
            
        # Save full frame snapshot with bounding box and ROI outline
        full_snap = frame.copy()
        cv2.polylines(full_snap, [np.array(self.roi_coords, dtype=np.int32)], True, (0, 0, 255), 2)
        cv2.rectangle(full_snap, (x1, y1), (x2, y2), (0, 0, 255), 3)
        cv2.putText(
            full_snap, 
            f"VIOLATION: ID {track_id} ({class_name})", 
            (x1, max(y1 - 15, 30)), 
            cv2.FONT_HERSHEY_SIMPLEX, 
            0.8, 
            (0, 0, 255), 
            2,
            cv2.LINE_AA
        )
        
        full_snap_filename = f"violation_{track_id}_frame_{frame_num}_{timestamp}.jpg"
        full_snap_path = os.path.join(EVIDENCE_DIR, full_snap_filename)
        cv2.imwrite(full_snap_path, full_snap)
        
        self.track_states[track_id]['snapshot_path'] = full_snap_path
        
    def record_violation_event(self, track_id, exit_frame, fps):
        """
        Computes duration and appends violation data to records list.
        """
        state = self.track_states[track_id]
        entry_frame = state['frame_entered']
        
        # Duration calculation based on frame differences and video FPS
        duration_sec = round((exit_frame - entry_frame) / fps, 2)
        exit_time_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
        
        record = {
            'Event ID': len(self.violation_records) + 1,
            'Track ID': track_id,
            'Vehicle Class': state['class_name'],
            'Entry Time': state['entry_time'],
            'Exit Time': exit_time_str,
            'Duration (Seconds)': duration_sec,
            'Entry Frame': entry_frame,
            'Exit Frame': exit_frame,
            'Snapshot Path': state.get('snapshot_path', 'N/A')
        }
        
        self.violation_records.append(record)
        print(f"[INFO] Logged violation for ID {track_id}: Duration = {duration_sec}s")
        
    def finalize_remaining_violations(self, final_frame, fps):
        """
        If video ends while a vehicle is still occupying the lane, close the log record.
        """
        for track_id, state in self.track_states.items():
            if state['violation_started']:
                self.record_violation_event(track_id, final_frame, fps)
                
    def save_report_to_csv(self, report_path):
        """
        Exports the recorded violations list to a CSV file.
        """
        if not self.violation_records:
            print("No violations recorded. Creating empty report template.")
            df = pd.DataFrame(columns=[
                'Event ID', 'Track ID', 'Vehicle Class', 'Entry Time', 'Exit Time', 
                'Duration (Seconds)', 'Entry Frame', 'Exit Frame', 'Snapshot Path'
            ])
        else:
            df = pd.DataFrame(self.violation_records)
            
        df.to_csv(report_path, index=False)
        print(f"Violation report exported to {report_path}")
        return df
        
    def save_frame_log_to_csv(self, log_path):
        """
        Exports the recorded frame occupancy history to a CSV file.
        """
        if not self.frame_records:
            df = pd.DataFrame(columns=[
                'Frame', 'Timestamp', 'Raw Occupancy (%)', 'Smoothed Occupancy (%)', 
                'Traffic Level', 'Vehicles in ROI'
            ])
        else:
            df = pd.DataFrame(self.frame_records)
            
        df.to_csv(log_path, index=False)
        print(f"Frame occupancy log exported to {log_path}")
        return df

## 4. Execution Pipeline

Run the cell below to define the driver function `run_pipeline`, which will orchestrate loading the video, instantiating the tracker/detector, running frame-by-frame inference, and saving the output video/CSV.

In [ ]:
def run_pipeline(video_path, output_path, roi_coords, model_name=MODEL_NAME, min_frames=MIN_VIOLATION_FRAMES, conf=CONFIDENCE_THRESHOLD, report_csv=REPORT_PATH, frame_log_csv=FRAME_LOG_PATH, low_thresh=LOW_OCCUPANCY_THRESHOLD, med_thresh=MEDIUM_OCCUPANCY_THRESHOLD, smooth_window=SMOOTHING_WINDOW_SIZE):
    if not os.path.exists(video_path):
        print(f"Error: Input video '{video_path}' does not exist. Please check your configuration path.")
        return
        
    print(f"Initializing video: {video_path}")
    cap = cv2.VideoCapture(video_path)
    
    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if fps == 0 or total_frames == 0:
        print("Error: Could not retrieve video properties. Check if the video codec is supported.")
        cap.release()
        return
        
    print(f"Video Info: Resolution={width}x{height}, FPS={fps:.2f}, Total Frames={total_frames}")
    
    # Initialize output video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Initialize Detector with new threshold and window configurations
    detector = EmergencyLaneDetector(
        model_name=model_name, 
        roi_coords=roi_coords, 
        min_violation_frames=min_frames, 
        conf=conf,
        low_threshold=low_thresh,
        medium_threshold=med_thresh,
        smoothing_window=smooth_window
    )
    
    frame_count = 0
    
    # Use tqdm progress bar
    progress_bar = tqdm(total=total_frames, desc="Processing Video Frames")
    
    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
                
            frame_count += 1
            
            # Skip frames based on sampling rate to speed up process if needed
            if frame_count % FRAME_SAMPLING_RATE != 0:
                progress_bar.update(1)
                continue
                
            # Process the frame
            annotated_frame = detector.process_frame(frame, frame_count, fps)
            
            # Write annotated frame to output file
            out.write(annotated_frame)
            
            progress_bar.update(1)
            
    except KeyboardInterrupt:
        print("\nProcessing interrupted by user. Finalizing logs and output video...")
    finally:
        # Finalize any ongoing violations at the end of the video
        detector.finalize_remaining_violations(frame_count, fps)
        
        # Cleanup video objects
        cap.release()
        out.release()
        cv2.destroyAllWindows()
        print("Video capture and output writer released.")
        
        # Save CSV reports
        detector.save_report_to_csv(report_csv)
        detector.save_frame_log_to_csv(frame_log_csv)
        
    print(f"Processing complete! Annotated video saved at: {output_path}")

## 5. Post-Processing Analytics & Visualization

After the video completes processing, you can use the cell below to load the generated CSV report, print summary stats, and plot graphical breakdowns of the violations.

In [ ]:
def visualize_results(report_path, frame_log_path=FRAME_LOG_PATH):
    if not os.path.exists(report_path):
        print(f"Report file '{report_path}' not found. Please run the pipeline first.")
        return
        
    df = pd.read_csv(report_path)
    
    # Set up 3 subplots: Class Distribution, Durations, and Lane Occupancy Timeline
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Pie chart of vehicle types
    if not df.empty:
        class_counts = df['Vehicle Class'].value_counts()
        axes[0].pie(class_counts, labels=class_counts.index, autopct='%1.1f%%', startangle=140, colors=['#ff9999','#66b3ff','#99ff99','#ffcc99'])
        axes[0].set_title("Violating Vehicle Class Distribution")
    else:
        axes[0].text(0.5, 0.5, 'No Violations Detected', ha='center', va='center')
        axes[0].set_title("Violating Vehicle Class Distribution")
        
    # 2. Histogram of violation durations
    if not df.empty:
        axes[1].hist(df['Duration (Seconds)'], bins=10, color='crimson', edgecolor='black', alpha=0.7)
        axes[1].set_xlabel("Duration (Seconds)")
        axes[1].set_ylabel("Number of Violations")
        axes[1].set_title("Distribution of Violation Durations")
    else:
        axes[1].text(0.5, 0.5, 'No Violations Detected', ha='center', va='center')
        axes[1].set_title("Distribution of Violation Durations")
        
    # 3. Line chart of lane occupancy percentage over time
    if os.path.exists(frame_log_path):
        df_frame = pd.read_csv(frame_log_path)
        if not df_frame.empty:
            axes[2].plot(df_frame['Frame'], df_frame['Smoothed Occupancy (%)'], color='royalblue', label='Smoothed Occupancy')
            axes[2].fill_between(df_frame['Frame'], df_frame['Smoothed Occupancy (%)'], color='royalblue', alpha=0.1)
            axes[2].plot(df_frame['Frame'], df_frame['Raw Occupancy (%)'], color='gray', linestyle='--', alpha=0.5, label='Raw Occupancy')
            axes[2].set_xlabel("Frame Number")
            axes[2].set_ylabel("Occupancy (%)")
            axes[2].set_title("Lane Occupancy Timeline")
            axes[2].legend()
            axes[2].grid(True, linestyle=':', alpha=0.6)
        else:
            axes[2].text(0.5, 0.5, 'No Occupancy Data', ha='center', va='center')
            axes[2].set_title("Lane Occupancy Timeline")
    else:
        axes[2].text(0.5, 0.5, f'{frame_log_path} not found', ha='center', va='center')
        axes[2].set_title("Lane Occupancy Timeline")
        
    plt.tight_layout()
    plt.show()

# To run visualization, uncomment the line below:
# visualize_results(REPORT_PATH)